# Section 5: Metadata Missingness and Usability Audit
Examines the demographic data targeting splits natively across operational lists.


In [1]:
import pandas as pd
import os
from IPython.display import display

OUTPUT_ROOT = r"D:\GradProj\Skin Cancer Dataset\pipeline_output"
d_audit = os.path.join(OUTPUT_ROOT, "audit_reports")
os.makedirs(d_audit, exist_ok=True)


## Load Restricted Operational Dataset


In [2]:
train_path = os.path.join(OUTPUT_ROOT, "manifests", "training_eligible_manifest.csv")
df_eval = pd.read_csv(train_path)
print(f"Loaded strict training targets natively: {len(df_eval)} pure rows.")


Loaded strict training targets natively: 20664 pure rows.


## Audit: Overall Database Missingness


In [3]:
fields = ["age_approx", "sex", "anatom_site_general", "lesion_id"]
total_rows = len(df_eval)

overall_missing = []
for f in fields:
    n_missing = df_eval[f].notna().sum()
    m_missing = total_rows - n_missing
    p_missing = round((m_missing / total_rows) * 100, 2)
    overall_missing.append({
        "field_name": f, "total_rows": total_rows,
        "non_missing_count": n_missing, "missing_count": m_missing,
        "missing_pct": p_missing
    })

df_overall = pd.DataFrame(overall_missing)
df_overall.to_csv(os.path.join(d_audit, "metadata_missingness_overall.csv"), index=False)
print("=== OVERALL METADATA MISSINGNESS ===")
display(df_overall)


=== OVERALL METADATA MISSINGNESS ===


,field_name,total_rows,non_missing_count,missing_count,missing_pct
0,age_approx,20664,20266,398,1.93
1,sex,20664,20316,348,1.68
2,anatom_site_general,20664,18386,2278,11.02
3,lesion_id,20664,18779,1885,9.12


## Audit: Missingness Sliced by Disease Class


In [4]:
class_missingness = []
for c in ["NV", "MEL", "BCC"]:
    df_c = df_eval[df_eval["final_authoritative_label"] == c]
    c_total = len(df_c)
    for f in fields:
        n_missing = df_c[f].notna().sum()
        m_missing = c_total - n_missing
        p_missing = round((m_missing / c_total) * 100, 2) if c_total > 0 else 0
        class_missingness.append({
            "field_name": f, "class_label": c, "class_total": c_total,
            "non_missing_count": n_missing, "missing_count": m_missing, "missing_pct": p_missing
        })

df_class_miss = pd.DataFrame(class_missingness)
df_class_miss.to_csv(os.path.join(d_audit, "metadata_missingness_by_class.csv"), index=False)
print("=== METADATA MISSINGNESS BY CLASS ===")
display(df_class_miss)


=== METADATA MISSINGNESS BY CLASS ===


,field_name,class_label,class_total,non_missing_count,missing_count,missing_pct
0,age_approx,NV,12851,12544,307,2.39
1,sex,NV,12851,12590,261,2.03
2,anatom_site_general,NV,12851,10768,2083,16.21
3,lesion_id,NV,12851,11303,1548,12.05
4,age_approx,MEL,4504,4419,85,1.89
5,sex,MEL,4504,4423,81,1.80
6,anatom_site_general,MEL,4504,4377,127,2.82
7,lesion_id,MEL,4504,4167,337,7.48
8,age_approx,BCC,3309,3303,6,0.18
9,sex,BCC,3309,3303,6,0.18


In [5]:
# =========================================================
# NEW CELL A — MISSINGNESS OVERLAP / CO-MISSINGNESS ANALYSIS
# Place this right after the "missingness by class" cell
# =========================================================

metadata_fields = ["age_approx", "sex", "anatom_site_general", "lesion_id"]

# 1) How many instances are missing each feature by disease
per_class_missing_counts = (
    df_eval.groupby("final_authoritative_label")[metadata_fields]
    .apply(lambda x: x.isna().sum())
    .reset_index()
)

print("=== MISSING VALUE COUNTS PER FEATURE BY DISEASE ===")
display(per_class_missing_counts)

# 2) Add row-level missingness flags
df_eval_overlap = df_eval.copy()

for col in metadata_fields:
    df_eval_overlap[f"{col}_missing"] = df_eval_overlap[col].isna()

# 3) Count how many metadata fields are missing per row
df_eval_overlap["num_missing_metadata_fields"] = df_eval_overlap[
    [f"{c}_missing" for c in metadata_fields]
].sum(axis=1)

print("=== NUMBER OF MISSING METADATA FIELDS PER ROW BY DISEASE ===")
missing_count_by_disease = (
    df_eval_overlap.groupby(["final_authoritative_label", "num_missing_metadata_fields"])
    .size()
    .reset_index(name="row_count")
    .sort_values(["final_authoritative_label", "num_missing_metadata_fields"])
)
display(missing_count_by_disease)

# 4) Build exact overlap pattern string for each row
def make_missing_pattern(row):
    missing_cols = [c for c in metadata_fields if pd.isna(row[c])]
    return "none_missing" if len(missing_cols) == 0 else "|".join(missing_cols)

df_eval_overlap["missing_pattern"] = df_eval_overlap.apply(make_missing_pattern, axis=1)

print("=== EXACT MISSINGNESS OVERLAP PATTERNS BY DISEASE ===")
missing_pattern_by_disease = (
    df_eval_overlap.groupby(["final_authoritative_label", "missing_pattern"])
    .size()
    .reset_index(name="row_count")
    .sort_values(["final_authoritative_label", "row_count"], ascending=[True, False])
)
display(missing_pattern_by_disease)

# 5) Save overlap reports
missing_count_by_disease.to_csv(
    os.path.join(d_audit, "metadata_missing_field_count_by_disease.csv"),
    index=False
)

missing_pattern_by_disease.to_csv(
    os.path.join(d_audit, "metadata_missing_overlap_patterns_by_disease.csv"),
    index=False
)

print("Saved:")
print("- metadata_missing_field_count_by_disease.csv")
print("- metadata_missing_overlap_patterns_by_disease.csv")

=== MISSING VALUE COUNTS PER FEATURE BY DISEASE ===


,final_authoritative_label,age_approx,sex,anatom_site_general,lesion_id
0,BCC,6,6,68,0
1,MEL,85,81,127,337
2,NV,307,261,2083,1548


=== NUMBER OF MISSING METADATA FIELDS PER ROW BY DISEASE ===


,final_authoritative_label,num_missing_metadata_fields,row_count
0,BCC,0,3235
1,BCC,1,68
2,BCC,2,6
3,MEL,0,4083
4,MEL,1,316
5,MEL,2,37
6,MEL,3,32
7,MEL,4,36
8,NV,0,9733
9,NV,1,2387


=== EXACT MISSINGNESS OVERLAP PATTERNS BY DISEASE ===


,final_authoritative_label,missing_pattern,row_count
2,BCC,none_missing,3235
1,BCC,anatom_site_general,68
0,BCC,age_approx|sex,6
10,MEL,none_missing,4083
9,MEL,lesion_id,245
7,MEL,anatom_site_general,71
5,MEL,age_approx|sex|anatom_site_general|lesion_id,36
6,MEL,age_approx|sex|lesion_id,32
8,MEL,anatom_site_general|lesion_id,20
4,MEL,age_approx|sex,13


Saved:
- metadata_missing_field_count_by_disease.csv
- metadata_missing_overlap_patterns_by_disease.csv


In [6]:
# =========================================================
# DROP ROWS WITH ALL 4 METADATA FIELDS MISSING (ALL CLASSES)
# WRITE A NEW POST-METADATA MANIFEST INSTEAD OF OVERWRITING
# =========================================================

metadata_fields = ["age_approx", "sex", "anatom_site_general", "lesion_id"]

# Identify rows where all metadata fields are missing
all_metadata_missing_mask = df_eval[metadata_fields].isna().all(axis=1)

all_metadata_missing_rows = df_eval[all_metadata_missing_mask].copy()

print("=== ROWS WITH ALL 4 METADATA FIELDS MISSING (ALL CLASSES) ===")
print(f"Count of rows with all metadata missing: {len(all_metadata_missing_rows)}")

# Show how many are being dropped per class
drop_counts_by_class = (
    all_metadata_missing_rows["final_authoritative_label"]
    .value_counts()
    .rename_axis("final_authoritative_label")
    .reset_index(name="drop_count")
)

print("=== DROP COUNTS BY DISEASE ===")
display(drop_counts_by_class)

review_cols = [
    "full_path",
    "file_name_with_extension",
    "final_authoritative_label",
    "age_approx",
    "sex",
    "anatom_site_general",
    "lesion_id"
]
available_review_cols = [c for c in review_cols if c in all_metadata_missing_rows.columns]

print("=== SAMPLE ROWS TO BE DROPPED ===")
display(all_metadata_missing_rows[available_review_cols].head(25))

# Save the rows being dropped for auditability
drop_path = os.path.join(d_audit, "all_metadata_missing_rows_dropped.csv")
all_metadata_missing_rows.to_csv(drop_path, index=False)

# Create a new post-metadata manifest instead of overwriting training_eligible_manifest.csv
df_eval_post_metadata = df_eval[~all_metadata_missing_mask].copy()

post_metadata_manifest_path = os.path.join(
    OUTPUT_ROOT,
    "manifests",
    "training_eligible_manifest_post_metadata.csv"
)
df_eval_post_metadata.to_csv(post_metadata_manifest_path, index=False)

print("=== POST-METADATA MANIFEST CREATED ===")
print(f"Dropped {len(all_metadata_missing_rows)} rows with all 4 metadata fields missing.")
print(f"Original training_eligible_manifest.csv row count: {len(df_eval)}")
print(f"Post-metadata manifest row count: {len(df_eval_post_metadata)}")
print(f"Dropped-row audit file saved to: {drop_path}")
print(f"New post-metadata manifest saved to: {post_metadata_manifest_path}")

=== ROWS WITH ALL 4 METADATA FIELDS MISSING (ALL CLASSES) ===
Count of rows with all metadata missing: 151
=== DROP COUNTS BY DISEASE ===


,final_authoritative_label,drop_count
0,NV,115
1,MEL,36


=== SAMPLE ROWS TO BE DROPPED ===


,full_path,file_name_with_extension,final_authoritative_label,age_approx,sex,anatom_site_general,lesion_id
57,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000008...,ISIC_0000080_downsampled.jpg,NV,NaN,NaN,NaN,NaN
58,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000008...,ISIC_0000081_downsampled.jpg,NV,NaN,NaN,NaN,NaN
59,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000008...,ISIC_0000082_downsampled.jpg,NV,NaN,NaN,NaN,NaN
60,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000008...,ISIC_0000085_downsampled.jpg,NV,NaN,NaN,NaN,NaN
61,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000008...,ISIC_0000086_downsampled.jpg,NV,NaN,NaN,NaN,NaN
62,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000008...,ISIC_0000087_downsampled.jpg,NV,NaN,NaN,NaN,NaN
63,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000008...,ISIC_0000088_downsampled.jpg,NV,NaN,NaN,NaN,NaN
64,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000008...,ISIC_0000089_downsampled.jpg,NV,NaN,NaN,NaN,NaN
65,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000009...,ISIC_0000091_downsampled.jpg,NV,NaN,NaN,NaN,NaN
66,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000009...,ISIC_0000092_downsampled.jpg,NV,NaN,NaN,NaN,NaN


=== POST-METADATA MANIFEST CREATED ===
Dropped 151 rows with all 4 metadata fields missing.
Original training_eligible_manifest.csv row count: 20664
Post-metadata manifest row count: 20513
Dropped-row audit file saved to: D:\GradProj\Skin Cancer Dataset\pipeline_output\audit_reports\all_metadata_missing_rows_dropped.csv
New post-metadata manifest saved to: D:\GradProj\Skin Cancer Dataset\pipeline_output\manifests\training_eligible_manifest_post_metadata.csv


In [7]:
print("=== POST-METADATA CLASS COUNTS ===")
post_metadata_class_counts = (
    df_eval_post_metadata["final_authoritative_label"]
    .value_counts()
    .rename_axis("final_authoritative_label")
    .reset_index(name="count")
)
display(post_metadata_class_counts)

=== POST-METADATA CLASS COUNTS ===


,final_authoritative_label,count
0,NV,12736
1,MEL,4468
2,BCC,3309


## Lesion ID Split Support Verification Computations


In [8]:
lesion_series = df_eval["lesion_id"].dropna()
unique_lesions = lesion_series.nunique()
lesion_counts = lesion_series.value_counts()
multiple_imm_lesions = (lesion_counts > 1).sum()
pct_lesions = round((len(lesion_series) / total_rows) * 100, 2)

lesion_report = [{
    "total_rows": total_rows,
    "rows_with_lesion_id": len(lesion_series),
    "rows_without_lesion_id": total_rows - len(lesion_series),
    "pct_rows_with_lesion_id": pct_lesions,
    "unique_lesion_ids": unique_lesions,
    "lesion_ids_with_multiple_images": multiple_imm_lesions
}]

df_lesion = pd.DataFrame(lesion_report)
df_lesion.to_csv(os.path.join(d_audit, "lesion_id_split_support_report.csv"), index=False)

print("=== LESION ID SPLIT SUPPORT EVIDENTIARY REPORT ===")
display(df_lesion)

print(f"Rows with lesion_id: {len(lesion_series)} / {total_rows} ({pct_lesions}%)")
print(f"Unique lesion IDs: {unique_lesions}")
print(f"Lesion IDs with multiple images: {multiple_imm_lesions}")

=== LESION ID SPLIT SUPPORT EVIDENTIARY REPORT ===


,total_rows,rows_with_lesion_id,rows_without_lesion_id,pct_rows_with_lesion_id,unique_lesion_ids,lesion_ids_with_multiple_images
0,20664,18779,1885,90.88,9735,3942


Rows with lesion_id: 18779 / 20664 (90.88%)
Unique lesion IDs: 9735
Lesion IDs with multiple images: 3942


## Generate Usage Classification Policy Structure


In [9]:
usage_policy = []

for f in fields:
    if f == "lesion_id":
        if pct_lesions >= 80 and multiple_imm_lesions > 0:
            usability_class = "usable_for_split_support"
        else:
            usability_class = "limited_use_only"
    else:
        f_pct = df_overall.loc[df_overall["field_name"] == f, "missing_pct"].values[0]

        if f_pct < 20:
            usability_class = "usable_for_audit"
        elif f_pct < 40:
            usability_class = "limited_use_only"
        else:
            usability_class = "not_usable"

    use_for_audit = usability_class in ["usable_for_audit", "usable_for_split_support", "limited_use_only"]
    use_for_reporting = usability_class in ["usable_for_audit", "usable_for_split_support", "limited_use_only"]
    use_for_split_support = usability_class == "usable_for_split_support"
    use_for_bias_analysis = (f in ["age_approx", "sex", "anatom_site_general"]) and (
        usability_class in ["usable_for_audit", "usable_for_split_support"]
    )

    usage_policy.append({
        "field_name": f,
        "usability_class": usability_class,
        "use_for_audit": use_for_audit,
        "use_for_reporting": use_for_reporting,
        "use_for_split_support": use_for_split_support,
        "use_for_bias_analysis": use_for_bias_analysis,
        "use_for_model_input_v1": False,
        "notes": f"Derived from observed missingness and split-support thresholds for this dataset state."
    })

df_policy = pd.DataFrame(usage_policy)
df_policy.to_csv(os.path.join(d_audit, "metadata_usage_policy.csv"), index=False)

print("=== METADATA USAGE POLICY EXPLICIT ASSIGNMENTS ===")
display(df_policy)

=== METADATA USAGE POLICY EXPLICIT ASSIGNMENTS ===


,field_name,usability_class,use_for_audit,use_for_reporting,use_for_split_support,use_for_bias_analysis,use_for_model_input_v1,notes
0,age_approx,usable_for_audit,True,True,False,True,False,Derived from observed missingness and split-su...
1,sex,usable_for_audit,True,True,False,True,False,Derived from observed missingness and split-su...
2,anatom_site_general,usable_for_audit,True,True,False,True,False,Derived from observed missingness and split-su...
3,lesion_id,usable_for_split_support,True,True,True,False,False,Derived from observed missingness and split-su...


## Missingness Drop Detection Array (No Auto-Drops)


In [10]:
# Prevent metadata missingness from dropping rows.
# This section only surfaces rows for inspection and review.

weak_metadata_flags = df_eval[
    df_eval["anatom_site_general"].isna() | df_eval["lesion_id"].isna()
].copy()

print(f"Identified {len(weak_metadata_flags)} rows lacking anatomical site or lesion_id.")
print("Strict policy enforcement: No rows are dropped during Section 5 metadata runs.")

missing_lesion_id = df_eval["lesion_id"].isna().sum()
missing_anatom_site = df_eval["anatom_site_general"].isna().sum()

print(f"Rows missing lesion_id: {missing_lesion_id}")
print(f"Rows missing anatom_site_general: {missing_anatom_site}")

review_cols = [
    "full_path",
    "final_authoritative_label",
    "age_approx",
    "sex",
    "anatom_site_general",
    "lesion_id"
]

available_review_cols = [c for c in review_cols if c in weak_metadata_flags.columns]

print("=== SAMPLE ROWS WITH WEAK STRUCTURAL METADATA ===")
display(weak_metadata_flags[available_review_cols].head(15))

Identified 3562 rows lacking anatomical site or lesion_id.
Strict policy enforcement: No rows are dropped during Section 5 metadata runs.
Rows missing lesion_id: 1885
Rows missing anatom_site_general: 2278
=== SAMPLE ROWS WITH WEAK STRUCTURAL METADATA ===


,full_path,final_authoritative_label,age_approx,sex,anatom_site_general,lesion_id
0,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000000...,NV,55.0,female,anterior torso,NaN
1,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000000...,NV,30.0,female,anterior torso,NaN
2,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000000...,NV,30.0,male,upper extremity,NaN
3,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000000...,NV,25.0,female,posterior torso,NaN
4,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000000...,NV,25.0,female,posterior torso,NaN
5,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000000...,NV,30.0,female,anterior torso,NaN
6,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000000...,NV,30.0,female,anterior torso,NaN
7,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000001...,NV,35.0,female,posterior torso,NaN
8,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000001...,NV,35.0,female,lower extremity,NaN
9,D:\GradProj\Skin Cancer Dataset\NV\ISIC_000001...,NV,30.0,male,posterior torso,NaN


## Export Usage Conclusion String


In [11]:
lesion_class = df_policy.loc[
    df_policy["field_name"] == "lesion_id", "usability_class"
].values[0]

summary = "Metadata Usability Summary\n\n"

summary += "1. Completeness Status:\n"
summary += "Overall metadata missingness was measured directly on training_eligible_manifest.csv and saved in metadata_missingness_overall.csv.\n\n"

summary += "2. Class-Specific Missingness:\n"
summary += "Per-class missingness for NV, MEL, and BCC was measured directly and saved in metadata_missingness_by_class.csv.\n\n"

summary += "3. Lesion ID Split Support:\n"
summary += (
    f"lesion_id was classified as {lesion_class} under the current rule because "
    f"{pct_lesions}% of eligible rows have lesion_id and "
    f"{multiple_imm_lesions} lesion IDs appear in multiple images.\n\n"
)

summary += "4. Metadata Usage Policy:\n"
summary += "Metadata remains audit/reporting/split-support only in v1 and is not allowed as model input.\n\n"

summary += "5. All-Metadata-Missing Drop Policy:\n"
summary += (
    f"{len(all_metadata_missing_rows)} rows across all classes with all four metadata fields missing "
    f"(age_approx, sex, anatom_site_general, lesion_id) were identified and written to "
    f"all_metadata_missing_rows_dropped.csv.\n"
)
summary += (
    f"A new dataset file, training_eligible_manifest_post_metadata.csv, was created with these rows removed. "
    f"The original training_eligible_manifest.csv was left unchanged.\n\n"
)


summary += "Allowed uses in v1:\n"
summary += "- audit\n"
summary += "- reporting\n"
summary += "- split support\n"
summary += "- bias analysis (only where supported by usability classification)\n\n"

summary += "Forbidden uses in v1:\n"
summary += "- model input\n"
summary += "- preprocessing decisions\n"
summary += "- augmentation control\n"
summary += "- label correction\n"

summary_path = os.path.join(d_audit, "metadata_usability_summary.txt")
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary)

print("Neutral artifact summary safely delivered to disk.")
print(f"Summary path: {summary_path}")
print("=== SUMMARY TEXT PREVIEW ===")
print(summary)

Neutral artifact summary safely delivered to disk.
Summary path: D:\GradProj\Skin Cancer Dataset\pipeline_output\audit_reports\metadata_usability_summary.txt
=== SUMMARY TEXT PREVIEW ===
Metadata Usability Summary

1. Completeness Status:
Overall metadata missingness was measured directly on training_eligible_manifest.csv and saved in metadata_missingness_overall.csv.

2. Class-Specific Missingness:
Per-class missingness for NV, MEL, and BCC was measured directly and saved in metadata_missingness_by_class.csv.

3. Lesion ID Split Support:
lesion_id was classified as usable_for_split_support under the current rule because 90.88% of eligible rows have lesion_id and 3942 lesion IDs appear in multiple images.

4. Metadata Usage Policy:
Metadata remains audit/reporting/split-support only in v1 and is not allowed as model input.

5. All-Metadata-Missing Drop Policy:
151 rows across all classes with all four metadata fields missing (age_approx, sex, anatom_site_general, lesion_id) were identi